# Phase 4.6 — SUPIR / HYPIR head-to-head at 4×

Apples-to-apples comparison on a 12-image subset of the frozen test set, at 4× upscale (the cap of suppixel.ai's SUPIR/HYPIR offerings).

**Subset selection** (see `outputs/supir/subset.json`) covers four buckets — clear wins for our pipeline (2), clear losses (2), 'stage-B-hurts' surprises (2), and category coverage (6). All 12 LR inputs are `data/test_images/{stem}_250.jpg` (the 4× variant: 250 → 1000).

**Methods compared at 4×:**
- `bicubic`, `lanczos` — pure interpolation baselines.
- `realesrgan` — non-diffusion SR baseline (RRDBNet GAN).
- `x4_upscaler` — base diffusion-SR model in our pipeline (no LoRA, no stage B).
- `two_stage` — full Phase 2 pipeline (x4-upscaler + ControlNet Tile img2img, no LoRA).
- `supir` — SUPIR Ultra mode, generic prompt, suppixel.ai.
- `hypir` — HYPIR generic prompt, suppixel.ai.
- `hypir_caption` — HYPIR with per-image BLIP-large captions (caption-quality A/B).

Generic prompt for everything except `hypir_caption`: `"a high-resolution photograph, sharp detail, professional quality"` — the same string our two-stage pipeline used during eval, so neither side gets a richer-caption boost.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = REPO_ROOT / "outputs" / "phase4_6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(REPO_ROOT / "outputs" / "eval" / "leaderboard_phase4_6.csv")
df = df[df.lr_size == 250]  # 4x rows only
subset = json.loads((REPO_ROOT / "outputs" / "supir" / "subset.json").read_text())
subset_images = [s["image"] for s in subset["subset"]]
df = df[df.image.isin(subset_images)]
print(
    f"loaded {len(df)} rows for {df.method.nunique()} methods x {df.image.nunique()} images at 4x"
)
df.head()

## 1. Mean LPIPS at 4× — the headline ranking

In [ ]:
method_order = [
    "hypir_caption",
    "hypir",
    "supir",
    "realesrgan",
    "x4_upscaler",
    "two_stage",
    "bicubic",
    "lanczos",
]
labels = {
    "hypir_caption": "HYPIR (per-image caption)",
    "hypir": "HYPIR (generic prompt)",
    "supir": "SUPIR Ultra (generic prompt)",
    "realesrgan": "Real-ESRGAN",
    "x4_upscaler": "SD x4-upscaler (stage A only)",
    "two_stage": "our two-stage (no LoRA)",
    "bicubic": "bicubic",
    "lanczos": "Lanczos",
}
agg = df.groupby("method")[["lpips", "dists"]].mean()
agg = agg.reindex(method_order)
print(agg.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(agg))
ax.barh(
    y,
    agg.lpips.values,
    color=[
        "#9C27B0"
        if m.startswith("hypir")
        else (
            "#673AB7"
            if m == "supir"
            else "#1976D2"
            if m in ("realesrgan", "x4_upscaler")
            else "#888"
        )
        for m in agg.index
    ],
)
ax.set_yticks(y)
ax.set_yticklabels([labels[m] for m in agg.index])
ax.invert_yaxis()  # best (lowest LPIPS) at top
ax.set_xlabel("mean LPIPS at 4× (lower = better)")
ax.set_title("Phase 4.6 head-to-head: mean LPIPS at 4× over 12 images")
for i, v in enumerate(agg.lpips.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "mean_lpips_bar.png", dpi=130, bbox_inches="tight")
plt.show()

## 2. Per-image side-by-side grids

One grid per image, showing center-crop at 100% zoom: HR / bicubic / Real-ESRGAN / x4-upscaler / our two-stage / SUPIR / HYPIR. Each cell is annotated with its LPIPS for that image.

In [ ]:
GRID_METHODS = [
    "bicubic",
    "realesrgan",
    "x4_upscaler",
    "two_stage",
    "supir",
    "hypir",
    "hypir_caption",
]
GRID_LABELS = {
    "bicubic": "bicubic",
    "realesrgan": "Real-ESRGAN",
    "x4_upscaler": "x4-upscaler",
    "two_stage": "our two-stage",
    "supir": "SUPIR",
    "hypir": "HYPIR",
    "hypir_caption": "HYPIR (caption)",
}
CROP_SIZE = 400
TARGET = 1000
PHASE1_CACHE = REPO_ROOT / "outputs" / "phase1" / "upscales"
PHASE2_CACHE = REPO_ROOT / "outputs" / "phase2" / "upscales"
SUPIR_DIR = REPO_ROOT / "outputs" / "supir"
HYPIR_DIR = REPO_ROOT / "outputs" / "hypir"


def method_path(method: str, stem: str) -> Path:
    if method == "two_stage":
        return PHASE2_CACHE / f"{stem}_250_d030_s20.jpg"
    if method in ("supir", "hypir"):
        return (SUPIR_DIR if method == "supir" else HYPIR_DIR) / f"{stem}_4x.jpg"
    if method == "hypir_caption":
        return HYPIR_DIR / f"{stem}_4x_caption.jpg"
    return PHASE1_CACHE / f"{stem}_250_{method}.jpg"


def render_image_grid(image_name: str, out_path: Path) -> None:
    stem = Path(image_name).stem
    cx = (TARGET - CROP_SIZE) // 2
    crop = (cx, cx, cx + CROP_SIZE, cx + CROP_SIZE)
    hr = Image.open(REPO_ROOT / "data" / "test_images" / image_name).convert("RGB").crop(crop)

    cols = ["HR"] + [GRID_LABELS[m] for m in GRID_METHODS]
    fig, axes = plt.subplots(1, len(cols), figsize=(2.4 * len(cols), 2.6))
    axes[0].imshow(hr)
    axes[0].set_title("HR (ref)", fontsize=10)
    axes[0].axis("off")

    for ax, m in zip(axes[1:], GRID_METHODS, strict=True):
        p = method_path(m, stem)
        if not p.is_file():
            ax.axis("off")
            ax.set_title(f"{GRID_LABELS[m]}\n(missing)", fontsize=9)
            continue
        img = Image.open(p).convert("RGB")
        if img.size != (TARGET, TARGET):
            img = img.resize((TARGET, TARGET), Image.Resampling.BICUBIC)
        ax.imshow(img.crop(crop))
        ax.axis("off")
        # Look up this method's LPIPS on this image
        row = df[(df.method == m) & (df.image == image_name)]
        score = row.lpips.iloc[0] if len(row) else float("nan")
        ax.set_title(f"{GRID_LABELS[m]}\nLPIPS {score:.3f}", fontsize=9)

    fig.suptitle(f"{stem}  —  center {CROP_SIZE}×{CROP_SIZE} crop, 4×", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close(fig)


for s in subset["subset"]:
    render_image_grid(s["image"], OUT_DIR / f"grid_{Path(s['image']).stem}.png")
print(f"{len(list(OUT_DIR.glob('grid_*.png')))} per-image grids in {OUT_DIR}")

In [ ]:
# Preview one grid inline
from IPython.display import Image as IPImage

IPImage(filename=str(OUT_DIR / "grid_Dubai_fine-arch.png"))

## 3. Caption-quality A/B on HYPIR

The same 12 images run through HYPIR twice — once with the generic prompt, once with the per-image BLIP-large caption. Since SUPIR was only run with the generic prompt (credit budget), this A/B is HYPIR-only.

**Hypothesis going in:** rich captions help diffusion-SR engines on visually-complex content (SUPIR's paper makes this claim explicit, motivating their use of LLaVA captions over BLIP).

**What we found:** the difference is essentially zero. See plot below.

In [ ]:
ab = (
    df[df.method.isin(["hypir", "hypir_caption"])]
    .pivot_table(index="image", columns="method", values="lpips")
    .reset_index()
)
ab["delta"] = ab.hypir_caption - ab.hypir  # negative = caption helped
ab = ab.merge(
    df[df.method == "hypir"][["image", "category", "challenges"]].drop_duplicates(),
    on="image",
)
print("=== Per-image: HYPIR caption vs generic ===")
print(
    ab[["image", "category", "challenges", "hypir", "hypir_caption", "delta"]]
    .sort_values("delta")
    .round(4)
    .to_string(index=False)
)
print("\nMean delta (caption - generic):", round(ab.delta.mean(), 4))
print("Caption better on N images:", int((ab.delta < 0).sum()), "/", len(ab))

fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(ab))
ab_sorted = ab.sort_values("delta").reset_index(drop=True)
colors = ["#2e7d32" if d < 0 else "#c62828" for d in ab_sorted.delta]
ax.barh(y, ab_sorted.delta.values, color=colors)
ax.set_yticks(y)
ax.set_yticklabels([Path(n).stem for n in ab_sorted.image])
ax.invert_yaxis()
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Δ LPIPS  (HYPIR caption − HYPIR generic)")
ax.set_title("HYPIR caption-quality A/B  (negative = BLIP caption helped)")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "caption_ab.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Takeaways

(Observations from the data above — these go into the Phase 7 writeup.)

### A. The SOTA gap to Real-ESRGAN at 4× is small

Mean LPIPS at 4× on the 12-image subset:
  - HYPIR: **0.254**, SUPIR: **0.289**, Real-ESRGAN: **0.297**, x4-upscaler: **0.306**.

On *this metric, at this ratio, on this subset*, SUPIR beats Real-ESRGAN by only ~3% LPIPS, and HYPIR (cheaper, suppixel's other tier) edges out SUPIR. This is narrower than SUPIR's reputation suggests. The main gap is structural — bicubic/Lanczos at 0.46–0.48 sit in their own much worse cluster — but inside the 'real SR methods' bucket, the differences are modest.

### B. Our base x4-upscaler is competitive; what we built on top is the regression

The x4-upscaler alone scores 0.306 LPIPS at 4× — within 6% of SUPIR. Stack ControlNet-Tile + img2img on top (our two-stage pipeline) and it jumps to 0.425 — a 39% regression from the same base. Our LoRA experiments compound this further (the broken x4-upscaler runs were 0.78–0.92).

**The diffusion-refinement-of-an-already-good-upscaler pattern overshoots when the LR is clean bicubic.** SUPIR's architecture works because it conditions on a heavily restored image and adds invented detail under tight zero-init adapters; our pipeline conditions on a stage-A output that's already near-correct, and stage-B's freedom to invent only adds noise.

### C. Per-image BLIP captions barely help HYPIR

Mean Δ LPIPS (caption − generic) across 12 HYPIR runs: ~0.000. The bar chart shows mixed direction by image, all small. **The 'rich captions matter' intuition didn't hold up empirically here.** Possible reasons:
  - HYPIR is less prompt-sensitive than SUPIR (which uses LLaVA captions deeply in its restoration encoder).
  - BLIP-large captions aren't 'rich enough' to differentiate vs generic — LLaVA outputs are visibly more detailed.
  - Our generic prompt (`a high-resolution photograph, sharp detail, professional quality`) is already a strong default for natural-image SR.

**Implication for the follow-on plan:** the cheap caption-upgrade path may be less valuable than expected for HYPIR-class models. SUPIR-with-LLaVA is a different question that this experiment didn't answer.

### D. Going beyond 4× has no available SOTA reference

Phase 4.6 is 4×-only because suppixel.ai's tools cap there. So do Real-ESRGAN x4plus and SD x4-upscaler. Our pipeline's 5× and 10× experiments (Phase 2) were therefore exploring beyond the boundary of what current public diffusion-SR tools support — the 'fidelity cliff' we observed at 10× isn't 'we lost to SUPIR at 10×' (we couldn't compare), it's a finding about extending diffusion-SR past its trained range. That belongs in the Phase 7 follow-on plan: the 5×/10× regime is research territory and a real next-project hook.